# Task 3 — Native categorical conditional quantiles
Compare fixed candidates on the same 5 folds, check the reused holdout, refit on Train only, then predict Test. This notebook uses project modules and cached baseline models; it is not standalone.

## 1. Setup

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Task3/models/hurdle_followup.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from Task3.models import native_mae_followup, raw_quantile_followup, mae_calibration, native_cat_quantile, finalize_native_cat
RESULTS = ROOT / "Task3/models/results"
RUN_TRAINING = False  # Results already trained; True reruns using caches.

## 2. Candidate experiments
Direct raw-THB MAE, raw positive quantiles, OOF gate/amount calibration, and native categorical CatBoost log-quantiles. Native CatBoost uses 900 trees, depth 5, learning rate 0.04 and L2=10; the occurrence gate stays unchanged. The fixed blend is 50% baseline and 50% new conditional prediction.

In [2]:
if RUN_TRAINING:
    native_mae_followup.run()
    raw_quantile_followup.run()
    mae_calibration.run()
    native_cat_quantile.run()

In [3]:
for name in ["native_mae_v1", "raw_quantile_v1", "mae_calibration_v1", "native_cat_quantile_v1"]:
    print(name)
    display(pd.read_csv(RESULTS / name / "cv_scores.csv"))
    display(json.loads((RESULTS / name / "selection.json").read_text()))

native_mae_v1
    fold  weight          MAE
0      1    0.00  2713.167940
1      1    0.25  2727.090041
2      1    0.50  2763.851826
3      1    1.00  2899.885620
4      2    0.00  2876.028077
5      2    0.25  2882.902794
6      2    0.50  2913.026926
7      2    1.00  3040.460221
8      3    0.00  2692.120110
9      3    0.25  2707.368242
10     3    0.50  2749.536375
11     3    1.00  2904.898151
12     4    0.00  2819.433892
13     4    0.25  2834.738327
14     4    0.50  2877.555399
15     4    1.00  3019.550834
16     5    0.00  2848.329657
17     5    0.25  2862.965226
18     5    0.50  2904.208531
19     5    1.00  3051.117765
{'baseline_MAE': 2789.815935343261, 'selected_weight': 0.0, 'candidate_means': {'0.0': 2789.815935343261, '0.25': 2803.012926049965, '0.5': 2841.6358113144865, '1.0': 2983.1825182642324}, 'fold_wins': {'0.25': 0, '0.5': 0, '1.0': 0}, 'selection_rule': 'MAE decrease >=5 THB, wins >=4/5; smallest mean among eligible; fixed recipes; reused folds'}
raw_quant

## 3. Holdout and final refit
OOF selection requires mean MAE gain ≥5 THB and wins in ≥4/5 folds. A nonpositive holdout gain vetoes export. These splits have been reused across experiments, so this is an exploratory comparison, not an untouched estimate or a guarantee of Kaggle improvement. Calibration evaluation is not fully nested. Test is read only after the choice and holdout review are fixed.

In [4]:
if RUN_TRAINING:
    display(finalize_native_cat.run())
OUT = RESULTS / "native_cat_quantile_v1"
display(json.loads((OUT / "holdout_review.json").read_text()))

{'selected': 'blend50', 'baseline_MAE': 2769.970711669546, 'candidate_MAE': 2756.4899648856112, 'rows': 20800, 'holdout_reused': True, 'gain_THB': 13.480746783934592, 'passed': True}


## 4. Submission verification
The exported combined CSV changes Task 3 only; Tasks 1, 2, 4 and 5 come from the existing 0.81047 submission. No Kaggle upload is performed. Re-running training produces a prediction payload; the verified CSV listed below was exported separately.

In [5]:
submission = ROOT / "submission/kaggle_t12345_t3_native_cat_quantile_v1_t4_efficientnet_blend50_v1_t5_directional_gap_cut_v1.csv"
print(submission)
if submission.exists():
    display(pd.read_csv(submission).head())
    display(json.loads((OUT / "submission_validation.json").read_text()))

/Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle/submission/kaggle_t12345_t3_native_cat_quantile_v1_t4_efficientnet_blend50_v1_t5_directional_gap_cut_v1.csv
         id  task1  task2        task3  task4  task5
0  ANS00001      1      3     0.000000      0      1
1  ANS00002      1      2  2326.604191      2      0
2  ANS00003      1      2     0.000000      3      1
3  ANS00004      1      1     0.000000      0      0
4  ANS00005      0      0     0.000000      3      0
{'rows': 25889, 'unique_ids': 25889, 'unchanged_columns': ['id', 'task1', 'task2', 'task4', 'task5'], 'task3_changed_rows': 12814, 'prediction_match': True, 'finite_nonnegative': True, 'submission': '/Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle/submission/kaggle_t12345_t3_native_cat_quantile_v1_t4_efficientnet_blend50_v1_t5_directional_gap_cut_v1.csv', 'sha256': '21d70b0fe7bfbc3531ccaf1075a1f53d052febb4c6c8293385cfdf2351beb551', 'uploaded': False}
